# Signature Verification with a Siamese CNN
**Deep Learning Lab | CEDAR | TensorFlow/Keras**

Train in Google Colab, then export the model to the Streamlit signature app.

**Before you start:** *Runtime → Change runtime type → T4 GPU*. On a CPU runtime it still works but is slower.

The question is not *"which class is this?"* (as in MNIST) but *"are these two signatures from the same person?"*.
A Siamese network uses **one** CNN twice: it turns each signature into a 128-number vector (an *embedding*) and
learns to put genuine signatures of the same person close together and forgeries far away.

## 1. Imports and reproducibility

In [ ]:
import re
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image, ImageFilter
from sklearn.metrics import roc_curve, roc_auc_score

SEED = 42
tf.keras.utils.set_random_seed(SEED)
rng = np.random.default_rng(SEED)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none (CPU runtime)")

## 2. Download the CEDAR dataset
CEDAR (University at Buffalo) contains **55 writers**, each with **24 genuine** signatures and **24 skilled forgeries**
(made by people who practiced copying the real signature). It is free for research and teaching. Download: ~254 MB.

In [ ]:
!wget -q -nc https://cedar.buffalo.edu/NIJ/data/signatures.rar
!apt-get -qq install -y unrar > /dev/null
!unrar x -o+ -inul signatures.rar data/

# Locate the folder that contains full_org/ and full_forg/
DATA_DIR = next(p.parent for p in Path("data").rglob("full_org") if (p.parent / "full_forg").exists())
print("Dataset folder:", DATA_DIR)

## 3. Preprocessing (identical to the Streamlit app)
This is the same code as `preprocessing.py`. The model must see images prepared **exactly** the way the app will prepare them.

1. Extract the ink: invert so ink is bright, then use an Otsu threshold to remove paper texture.
2. Crop to the signature, scale it into a 128×256 frame (keeping the aspect ratio) and center it.
3. Apply a light blur so the model tolerates different pen widths.

In [ ]:
OUT_H, OUT_W = 128, 256


def _otsu(gray):
    hist = np.bincount(gray.ravel(), minlength=256).astype(np.float64)
    total = hist.sum()
    cum = np.cumsum(hist)
    cum_mean = np.cumsum(hist * np.arange(256))
    w0 = cum / total
    w1 = 1.0 - w0
    mu0 = cum_mean / np.maximum(cum, 1)
    mu1 = (cum_mean[-1] - cum_mean) / np.maximum(total - cum, 1)
    return int(np.argmax(w0 * w1 * (mu0 - mu1) ** 2))


def ink_map(image_data):
    data = np.asarray(image_data)
    if data.ndim == 3 and data.shape[2] == 4:
        gray = data[:, :, :3].astype(np.float32).mean(axis=2)
        alpha = data[:, :, 3].astype(np.float32) / 255.0
        return np.clip((255.0 - gray) * alpha, 0, 255).astype(np.uint8)
    gray = data[:, :, :3].astype(np.float32).mean(axis=2) if data.ndim == 3 else data.astype(np.float32)
    if gray.max() <= 1.0:
        gray *= 255.0
    gray = np.clip(gray, 0, 255).astype(np.uint8)
    if np.median(gray) > 127:
        gray = 255 - gray
    t = _otsu(gray)
    return np.where(gray > t, gray, 0).astype(np.uint8)


def prepare_signature(image_data):
    ink = ink_map(image_data)
    ys, xs = np.where(ink > 30)
    if len(xs) < 20:
        return None, None
    x0, x1, y0, y1 = xs.min(), xs.max() + 1, ys.min(), ys.max() + 1
    cropped = Image.fromarray(ink[y0:y1, x0:x1], mode="L")
    w, h = cropped.size
    margin = 8
    scale = min((OUT_W - 2 * margin) / w, (OUT_H - 2 * margin) / h)
    rw, rh = max(1, round(w * scale)), max(1, round(h * scale))
    resized = cropped.resize((rw, rh), Image.Resampling.LANCZOS)
    frame = Image.new("L", (OUT_W, OUT_H), 0)
    frame.paste(resized, ((OUT_W - rw) // 2, (OUT_H - rh) // 2))
    frame = frame.filter(ImageFilter.GaussianBlur(1.2))
    return np.asarray(frame, dtype=np.float32) / 255.0, frame

## 4. Load and explore the data

In [ ]:
def load_folder(folder, prefix):
    images = {}
    for path in sorted(Path(folder).glob(f"{prefix}_*_*.png")):
        writer = int(re.match(rf"{prefix}_(\d+)_", path.name).group(1))
        arr, _ = prepare_signature(np.asarray(Image.open(path).convert("L")))
        if arr is not None:
            images.setdefault(writer, []).append((arr * 255).astype(np.uint8))
    return {w: np.stack(v) for w, v in images.items()}


genuine = load_folder(DATA_DIR / "full_org", "original")
forged = load_folder(DATA_DIR / "full_forg", "forgeries")
writers = sorted(set(genuine) & set(forged))
print(f"Writers: {len(writers)}")
print(f"Genuine images: {sum(len(genuine[w]) for w in writers)} | Forged images: {sum(len(forged[w]) for w in writers)}")
print("One preprocessed image:", genuine[writers[0]][0].shape)

In [ ]:
raw = Image.open(sorted((DATA_DIR / "full_org").glob("original_*_1.png"))[0])
fig, ax = plt.subplots(1, 2, figsize=(10, 2.5))
ax[0].imshow(raw, cmap="gray"); ax[0].set_title("Raw scan")
ax[1].imshow(genuine[writers[0]][0], cmap="gray"); ax[1].set_title("After preprocessing (128×256)")
for a in ax: a.axis("off")
plt.show()

show = writers[:3]
fig, ax = plt.subplots(len(show), 4, figsize=(14, 2.2 * len(show)))
for row, w in enumerate(show):
    for col, (img, title) in enumerate([(genuine[w][0], "genuine"), (genuine[w][1], "genuine"),
                                        (forged[w][0], "FORGERY"), (forged[w][1], "FORGERY")]):
        ax[row, col].imshow(img, cmap="gray")
        ax[row, col].set_title(f"Writer {w} · {title}")
        ax[row, col].axis("off")
plt.tight_layout(); plt.show()

**Question:** looking at the grid, which differences between genuine and forged signatures can you see? Which are hard to see?

## 5. Writer-independent split
We hold back **10 whole writers** for testing. The model never sees them during training, exactly like a new person
who enrolls in the app. Splitting by image instead would let the model memorize each person and give an over-optimistic score.

In [ ]:
TEST_WRITERS = 10
shuffled = list(writers)
rng.shuffle(shuffled)
test_writers, train_writers = shuffled[:TEST_WRITERS], shuffled[TEST_WRITERS:]
print("Train writers:", len(train_writers), "| Test writers:", len(test_writers))
print("Test writers:", sorted(test_writers))

## 6. Build pairs
Each training example is a **pair** of signatures with a label:

| Pair | Label | Share |
|---|---|---|
| genuine + genuine, same writer | 1 (same) | 50% |
| genuine + skilled forgery of that writer | 0 (different) | 25% |
| genuine + genuine of another writer ("random forgery") | 0 (different) | 25% |

New random pairs are drawn every epoch, so the model sees many more combinations than there are images.

In [ ]:
KINDS = ["genuine", "skilled forgery", "other writer"]


def make_pairs(writer_ids, n):
    a, b, y, kinds = [], [], [], []
    for i in range(n):
        w = writer_ids[rng.integers(len(writer_ids))]
        g = genuine[w]
        a.append(g[rng.integers(len(g))])
        kind = 0 if i % 4 < 2 else i % 4 - 1
        if kind == 0:
            b.append(g[rng.integers(len(g))])
        elif kind == 1:
            b.append(forged[w][rng.integers(len(forged[w]))])
        else:
            other = writer_ids[rng.integers(len(writer_ids))]
            while other == w:
                other = writer_ids[rng.integers(len(writer_ids))]
            b.append(genuine[other][rng.integers(len(genuine[other]))])
        y.append(1.0 if kind == 0 else 0.0)
        kinds.append(kind)
    to_x = lambda x: (np.stack(x).astype(np.float32) / 255.0)[..., np.newaxis]
    return to_x(a), to_x(b), np.array(y, dtype=np.float32), np.array(kinds)


val_a, val_b, val_y, val_kind = make_pairs(test_writers, 2000)
print("Validation pairs:", val_a.shape, "| share labelled 'same':", val_y.mean())

## 7. Build the Siamese network
- **Embedding CNN:** 4 blocks of convolution + batch normalization + max pooling, then global average pooling
  and a 128-unit dense layer. The output is normalized to length 1.
- **Siamese model:** the *same* embedding CNN (shared weights) processes both images; the output is the Euclidean distance.
- **Contrastive loss:** for genuine pairs it pushes the distance toward 0; for different pairs it pushes the distance
  above a margin of 1.

In [ ]:
def build_embedding():
    inp = tf.keras.layers.Input(shape=(OUT_H, OUT_W, 1))
    x = inp
    for filters in (32, 64, 128, 128):
        x = tf.keras.layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = tf.keras.layers.BatchNormalization()(x)
        x = tf.keras.layers.MaxPooling2D(2)(x)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(128)(x)
    out = tf.keras.layers.UnitNormalization()(x)
    return tf.keras.Model(inp, out, name="signature_embedding")


class EuclideanDistance(tf.keras.layers.Layer):
    def call(self, a, b):
        return tf.sqrt(tf.reduce_sum(tf.square(a - b), axis=1, keepdims=True) + 1e-9)


def contrastive_loss(y, d, margin=1.0):
    return tf.reduce_mean(y * tf.square(d) + (1 - y) * tf.square(tf.maximum(margin - d, 0.0)))


embedding = build_embedding()
in_a = tf.keras.layers.Input(shape=(OUT_H, OUT_W, 1))
in_b = tf.keras.layers.Input(shape=(OUT_H, OUT_W, 1))
distance = EuclideanDistance()(embedding(in_a), embedding(in_b))
siamese = tf.keras.Model([in_a, in_b], distance, name="siamese")
siamese.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss=contrastive_loss)
embedding.summary()

## 8. Train the model
About 1 minute per epoch on a T4 GPU.

In [ ]:
EPOCHS = 15
PAIRS_PER_EPOCH = 6000
history = {"loss": [], "val_loss": []}
for epoch in range(EPOCHS):
    tr_a, tr_b, tr_y, _ = make_pairs(train_writers, PAIRS_PER_EPOCH)  # fresh pairs every epoch
    h = siamese.fit([tr_a, tr_b], tr_y, validation_data=([val_a, val_b], val_y),
                    batch_size=32, epochs=1, verbose=0)
    for k in history:
        history[k].append(h.history[k][0])
    print(f"Epoch {epoch + 1}/{EPOCHS} · loss {history['loss'][-1]:.4f} · val_loss {history['val_loss'][-1]:.4f}")

## 9. Learning curves

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(history["loss"], label="train")
plt.plot(history["val_loss"], label="validation (unseen writers)")
plt.title("Contrastive loss"); plt.xlabel("Epoch"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

## 10. Evaluate on unseen writers
- **FAR** (false acceptance rate): share of forgeries accepted as genuine.
- **FRR** (false rejection rate): share of genuine signatures rejected.
- **EER** (equal error rate): the error at the threshold where FAR = FRR. Lower is better.

In [ ]:
d = siamese.predict([val_a, val_b], verbose=0).ravel()
fpr, tpr, thr = roc_curve(val_y, -d)  # smaller distance = more likely genuine
fnr = 1 - tpr
best = int(np.nanargmin(np.abs(fnr - fpr)))
THRESHOLD = float(-thr[best])
accept = d <= THRESHOLD
metrics = {
    "test_writers": len(test_writers),
    "test_pairs": int(len(val_y)),
    "threshold": THRESHOLD,
    "accuracy": float(np.mean(accept == (val_y == 1))),
    "eer": float((fpr[best] + fnr[best]) / 2),
    "auc": float(roc_auc_score(val_y, -d)),
    "false_reject_rate": float(np.mean(~accept[val_kind == 0])),
    "far_skilled_forgery": float(np.mean(accept[val_kind == 1])),
    "far_other_writer": float(np.mean(accept[val_kind == 2])),
}
print(json.dumps(metrics, indent=2))

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for k, color in zip(range(3), ["tab:green", "tab:red", "tab:orange"]):
    ax[0].hist(d[val_kind == k], bins=40, alpha=0.55, color=color, label=KINDS[k])
ax[0].axvline(THRESHOLD, color="black", ls="--", label=f"threshold {THRESHOLD:.3f}")
ax[0].set_title("Distance between the two signatures of a pair"); ax[0].set_xlabel("Distance"); ax[0].legend()
ax[1].plot(fpr, tpr); ax[1].plot([0, 1], [0, 1], ls=":", color="gray")
ax[1].scatter(fpr[best], tpr[best], color="black", zorder=3, label=f"EER {metrics['eer']:.1%}")
ax[1].set_title(f"ROC curve · AUC {metrics['auc']:.3f}"); ax[1].set_xlabel("FAR"); ax[1].set_ylabel("1 − FRR")
ax[1].legend(); ax[1].grid(alpha=0.3)
plt.show()

**Read the histogram:** green pairs should sit to the left of the threshold and red/orange pairs to the right.
Overlap means errors. Skilled forgeries (red) are usually the hardest. Why?

## 11. Visualize mistakes

In [ ]:
errors = np.where(accept != (val_y == 1))[0][:8]
fig, ax = plt.subplots(len(errors), 2, figsize=(9, 1.9 * max(len(errors), 1)), squeeze=False)
for row, i in enumerate(errors):
    verdict = "accepted" if accept[i] else "rejected"
    ax[row, 0].imshow(val_a[i, :, :, 0], cmap="gray"); ax[row, 0].set_title("Reference")
    ax[row, 1].imshow(val_b[i, :, :, 0], cmap="gray")
    ax[row, 1].set_title(f"{KINDS[val_kind[i]]} → {verdict} (d={d[i]:.3f})")
    for a in ax[row]: a.axis("off")
plt.tight_layout(); plt.show()

## 12. Export the model
Save the **embedding CNN only** (not the Siamese wrapper). The app computes an embedding for each enrolled reference
and for the new signature, then compares distances. Place both files in `model/` in the signature app folder:

- `signature_embedding.keras`
- `siamese_metrics.json` (contains the decision threshold)

In [ ]:
embedding.save("signature_embedding.keras")
Path("siamese_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print("Saved signature_embedding.keras and siamese_metrics.json")
# Colab only: uncomment to download the files
# from google.colab import files
# files.download("signature_embedding.keras")
# files.download("siamese_metrics.json")

## 13. Reflect
1. Compare **FAR for skilled forgeries** with **FAR for other writers**. Which forgery is easier to detect, and why?
2. Why must the test writers be people the model has never seen?
3. CEDAR signatures are scanned ink on paper; the app also receives tablet drawings. What effect could this have?
   How could you check it?
4. Is the EER good enough for a bank? What else would a real system need (pen pressure, speed, more references, a human check)?